# Combine Multiple Stance Datasets Into One

**Goal:** standardize several raw labeled datasets into the same `conversations`
schema, then merge them into a single dataset *before* fine-tuning.

**Why combine-then-train-once, not sequential fine-tuning:** per Unsloth's dataset
guide, "training an already fine-tuned model can potentially alter the quality and
knowledge acquired during the previous fine-tuning process." So if you have 2+ stance
datasets, standardize + concatenate them first, then run `finetune_stance.py` once on
the combined set -- don't fine-tune on dataset A, then fine-tune that checkpoint again
on dataset B.

**Self-contained on purpose**, same as notebook 1: the prompt/label logic is defined
directly below, not imported from `prepare_stance_data.py`, so the format is visible
while you're learning it rather than hidden behind a function call.


## 1. Define the prompt template + label map

Same format as notebook 1 -- `STANCE_PROMPT_TEMPLATE` must match the Tilburg eval
prompt exactly, `STANCE_LABEL_MAP` canonicalizes whatever label spelling each raw
dataset happens to use onto `FAVOR` / `AGAINST` / `NONE`.


In [7]:
STANCE_PROMPT_TEMPLATE = (
    "Stance classification is the task of determining the expressed or implied opinion, "
    "or stance, of a document toward a certain, specified target. "
    "Analyze the following document and determine its stance toward the provided query.\n\n"
    "QUERY: {target}\n\n"
    "DOCUMENT: {text}\n\n"
    'Return valid JSON in exactly this format: {{"stance": "FAVOR"}}\n'
    'The "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\n'
    'Use "FAVOR" only when the author is definitely in favor of the query. '
    'Use "AGAINST" only when the author is definitely against the query. '
    'Use "NONE" if any of the following holds: (a) the document does not discuss the query '
    "at all, (b) the document discusses it but the author takes no clear side "
    "(neutral/balanced), or (c) the author's position cannot be determined with confidence. "
    "Do not guess from indirect hints.\n"
)

# Maps every label spelling seen across stance datasets onto our fixed 3-way vocabulary.
# Add an entry here whenever a new dataset uses a label spelling we haven't seen yet --
# canonicalize_label() below deliberately raises instead of silently guessing.
STANCE_LABEL_MAP = {
    "FAVOR": "FAVOR",
    "AGAINST": "AGAINST",
    "NONE": "NONE",
    "PRO": "FAVOR",
    "NEUTRAL": "NONE",
    "UNCLEAR": "NONE",
    "UNRELATED": "NONE",
    "SUPPORTS": "FAVOR",
    "DENIES": "AGAINST",
}


def canonicalize_label(raw_label):
    normalized = str(raw_label).strip().upper()
    if normalized not in STANCE_LABEL_MAP:
        raise ValueError(
            f"Unrecognized stance label {raw_label!r} -- add it to STANCE_LABEL_MAP "
            f"(known: {sorted(STANCE_LABEL_MAP)})"
        )
    return STANCE_LABEL_MAP[normalized]


def build_conversations(df, text_column, target_column, label_column):
    conversations = []
    for _, row in df.iterrows():
        prompt = STANCE_PROMPT_TEMPLATE.format(target=row[target_column], text=row[text_column])
        answer = json.dumps({"stance": canonicalize_label(row[label_column])})
        conversations.append(
            [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
            ]
        )
    return conversations


## 2. Define your dataset sources

Each raw dataset can use different column names for text/target/label -- list them
here and we normalize each one to what `build_conversations()` expects. The second
entry below is a placeholder: replace `path` and the three column names with your
actual second dataset.


In [6]:
from pathlib import Path
import json
import pandas as pd

SOURCES = [
    {
        "name": "semeval2016_task6",
        "path": Path(
            "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
            "/data_in/semeval/semeval2016-task6-trainingdata.txt"
        ),
        "read_kwargs": {"sep": "\t", "encoding": "latin1"},
        "text_column": "Tweet",
        "target_column": "Target",
        "label_column": "Stance",
    },
    {
        "name": "mtcsd",
        "path": Path(
            "/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper"
            "/data_in/mtcsd/mtcsd_train.csv"
        ),
        "read_kwargs": {},  # plain UTF-8, comma-separated -- pandas defaults are fine
        "text_column": "content",
        "target_column": "query",
        "label_column": "stance_label",  # already favor/against/none -- canonicalize_label() upper-cases it
    },
]


## 3. Load + build conversations per source

Kept separate for now (not concatenated yet) so we can check label balance per source
before merging.


In [9]:
from datasets import Dataset
per_source_datasets = {}

for src in SOURCES:
    if not src["path"].exists():
        print(f"Skipping {src['name']!r} -- path not found: {src['path']}")
        continue

    if src["path"].suffix == ".jsonl":
        df = pd.read_json(src["path"], lines=True)
    elif src["path"].suffix == ".json":
        df = pd.read_json(src["path"])
    else:
        df = pd.read_csv(src["path"], **src["read_kwargs"])

    conversations = build_conversations(
        df, src["text_column"], src["target_column"], src["label_column"]
    )
    per_source_datasets[src["name"]] = Dataset.from_dict({"conversations": conversations})
    print(f"{src['name']}: {len(conversations)} examples")


/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nityaakalra/Desktop/fine_tune/fine-tuning-slms/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


semeval2016_task6: 2814 examples
mtcsd: 13134 examples


## 4. Check label balance across sources

Unsloth's guidance: keep data "balanced in many areas so your model does not
overfit" -- don't let one source's label skew dominate the combined set.


In [10]:
import json
from collections import Counter

for name, ds in per_source_datasets.items():
    labels = [json.loads(convo[1]["content"])["stance"] for convo in ds["conversations"]]
    print(name, Counter(labels))


semeval2016_task6 Counter({'AGAINST': 1342, 'NONE': 741, 'FAVOR': 731})
mtcsd Counter({'NONE': 6345, 'AGAINST': 3979, 'FAVOR': 2810})


## 5. Combine into one dataset


In [11]:
from datasets import Dataset, concatenate_datasets

combined = concatenate_datasets(list(per_source_datasets.values()))
combined = combined.shuffle(seed=42)
print(f"Combined: {len(combined)} examples from {len(per_source_datasets)} sources")


Combined: 15948 examples from 2 sources


## 6. Drop exact duplicate conversations (can happen if sources overlap)


In [12]:
seen = set()
keep_indices = []
for i, convo in enumerate(combined["conversations"]):
    key = json.dumps(convo, sort_keys=True)
    if key not in seen:
        seen.add(key)
        keep_indices.append(i)

deduped = combined.select(keep_indices)
print(f"Removed {len(combined) - len(deduped)} exact duplicates -> {len(deduped)} examples")
combined = deduped


Removed 4 exact duplicates -> 15944 examples


## 7. Sanity check + preview


In [13]:
combined


Dataset({
    features: ['conversations'],
    num_rows: 15944
})

In [14]:
combined[0]  # inspect one merged example


{'conversations': [{'content': 'Stance classification is the task of determining the expressed or implied opinion, or stance, of a document toward a certain, specified target. Analyze the following document and determine its stance toward the provided query.\n\nQUERY: Bitcoin\n\nDOCUMENT: Or, hear me out, pass regulations that require CEXs to have those in place\n\nReturn valid JSON in exactly this format: {"stance": "FAVOR"}\nThe "stance" value must be exactly one of: "FAVOR", "AGAINST", "NONE".\nUse "FAVOR" only when the author is definitely in favor of the query. Use "AGAINST" only when the author is definitely against the query. Use "NONE" if any of the following holds: (a) the document does not discuss the query at all, (b) the document discusses it but the author takes no clear side (neutral/balanced), or (c) the author\'s position cannot be determined with confidence. Do not guess from indirect hints.\n',
   'role': 'user'},
  {'content': '{"stance": "FAVOR"}', 'role': 'assistan

## 8. Push the combined dataset to the Hub

Use a **new** repo id -- don't overwrite the semeval-only dataset from notebook 1.


In [15]:
PUSH_TO_HUB_ID = "nityaak/stance-combined-v1"
PRIVATE = True

# Uncomment when ready:
combined.push_to_hub(PUSH_TO_HUB_ID, private=PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{PUSH_TO_HUB_ID}")


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  9.69ba/s]
Processing Files (1 / 1): 100%|██████████| 2.99MB / 2.99MB,  263kB/s  
New Data Upload: 100%|██████████| 2.99MB / 2.99MB,  263kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.44s/ shards]


Pushed to https://huggingface.co/datasets/nityaak/stance-combined-v1


## Recap

- Combine **before** training; don't sequentially fine-tune on dataset A then dataset B
- Same `conversations` schema + same prompt/label logic as notebook 1 -- copied inline
  in both notebooks on purpose, for learning; `prepare_stance_data.py` is the copy the
  real pipeline runs off of
- Check per-source label balance before merging
- Dedupe overlapping examples across sources
